In [4]:
from vertex_lite.custom import load_data,load_data_v3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import glob
import os
import numpy as np

## Load Sort and Filter

In [5]:
def filter_data(time,local=False,cluster=False):
    if cluster:
        rpath='resultados_cluster'
    else:
        rpath='resultados'
    if local:
        df=load_data_v3(f'{rpath}/noise_*/pct_*/sim_*/mutant.parquet',columns=['time','shape_neigh'],time_filter=time)
        df=df.groupby(['mesh_type','pct_mutant'])['shape_neigh'].mean().reset_index()
    else:
        df=load_data_v3(f'{rpath}/noise_*/pct_*/sim_*/global.parquet',columns=['time','shape_global'],time_filter=time)
        df=df.groupby(['mesh_type','pct_mutant'])['shape_global'].mean().reset_index()
    if df.empty:
        
        return print(f"No data found for time {time} and local {local}")
    

    return df


In [6]:
df=load_data_v3(f'resultados/noise_0.45/pct_0.0/sim_0/mutant.parquet',columns=['time','shape_neigh'],time_filter=10)
print(df)
df=df.groupby(['mesh_type','pct_mutant'])['shape_neigh'].mean().reset_index()
df.head(100)

   time  shape_neigh  mesh_type  pct_mutant  sim_id
0    10            0       0.45         0.0       0


,mesh_type,pct_mutant,shape_neigh
0,0.45,0.0,0.0


## Plotting

In [7]:
#plot_heatmap(time, local, "hex")
from fileinput import filename


def get_heatmap_data(time, local=False, case=None,cluster=False):
    df = filter_data(time, local, cluster)
    shape_col = 'shape_neigh' if local else 'shape_global'
    if case == "shape":
            title = "Shape Index"
            # Pivotea directamente: mesh_type como columnas, pct_mutant como índice, valor como shape index
            heatmap_data = df.set_index(['mesh_type', 'pct_mutant'])[shape_col].unstack()

    heatmap_data = heatmap_data.sort_index().sort_index(axis=1)
    heatmap_data = heatmap_data.T
    return heatmap_data, title
import matplotlib.colors as mcolors
def plot_heatmap(heatmap_data,title,time, local=False, case=None,cluster=False):
    cmap = sns.diverging_palette(220, 20, as_cmap=True)
    norm = mcolors.TwoSlopeNorm(vmin=3.72, vcenter=3.81, vmax=3.9)

    plt.figure(figsize=(10, 8))
    sns.set_theme(style="white", context="talk")

    plt.rcParams.update({
        "font.size": 20,
        "axes.labelsize": 25,
        "axes.titlesize": 25,
        "legend.fontsize": 12,
        "axes.linewidth": 0.0,
        "xtick.labelsize": 24,
        "ytick.labelsize": 24,
    })
    # ---- PLOT WITH IMSHOW ----
    fig, ax = plt.subplots(figsize=(7, 5))

    # Convert DataFrame to numpy array for imshow
    # Note: imshow handles the orientation differently than sns, 
    # so we use the transposed/unstacked data directly.
    im = ax.imshow(
        heatmap_data.values, 
        cmap=cmap, 
        origin='lower',  # Puts the first row at the bottom (standard for plots)
        aspect='auto',   # 'equal' makes cells square, 'auto' fills the figure
        norm=norm
    )

    # --- AXES LABELS ---
    # In imshow, we must manually set the ticks and labels
    step = 2  
    plt.tick_params(axis='both', labelsize = 15)
    # X-axis: Mesh Distortion (Columns)
    x_ticks = np.arange(0, len(heatmap_data.columns), step)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(heatmap_data.columns[::step], rotation=45)

    # Y-axis: % mutants (Index)
    y_ticks = np.arange(0, len(heatmap_data.index), step)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(heatmap_data.index[::step])

    # --- COLORBAR & LABELS ---
    
    # --- COLORBAR ---
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(case)

    # Annotations and Titles
    ax.annotate(
        f'T={time} | local={local}',
        xy=(0.0, -0.2),
        xycoords='axes fraction',
        ha='left',
        fontsize=10
    )

    # Labels (Standard Matplotlib)
    plt.xlabel("Initial Mesh Distortion")
    plt.ylabel("% mutants")
    plt.title(f'{title} for Neighbors of Mutant Cells' if local else f'{title} for Whole Tissue')
    plt.xlabel("Initial Mesh Distortion")
    plt.ylabel("% mutants")
    plt.title(f'{title} for Neighbors of Mutant Cells' if local else f'{title} for Whole Tissue')

    plt.tight_layout()
    
    suffix = "local" if local else "global"
    rpath='resultados_cluster' if cluster else 'resultados'
    clustering= "cluster" if cluster else "random"
    
    savedir = f"{rpath}/figures/paper/si_heatmaps/{case}"
    savedir="figures_paper_final/si_heatmaps"
    savedir = "figures_paper/si_heatmaps_2"
    os.makedirs(savedir, exist_ok=True)

    filename = f"{savedir}/heatmap_{clustering}_time_{time}_{case}_{suffix}.svg"

    fig.savefig(filename, format ='svg')
    plt.close(fig)

## Figures

In [8]:
# times=[[0,100,250,1995], [0,1000,2500,4000]]
# times=[[0, 4995], [0, 4995]]
times = [[1995], [14995]]
# for c in ['shape']:
#     for l in [True, False]:
#         for t in times[0]:
           
#             plot_heatmap_v2(time=t, local=l, case=c,cluster=False)
#         for t in times[1]:
          
#             plot_heatmap_v2(time=t, local=l, case=c,cluster=True)


In [9]:
data_cache = {}

# for c in ["hex", "std"]:
for c in ["shape"]: # we are only going to use the deviation from the hexagon
    for l in [True, False]:
        # Process times for local
        for t in times[0]:
            key = (t, l, c, False) # (time, local, case, cluster)
            print(f"Analyzing {key}...")
            data_cache[key] = get_heatmap_data(time=t, local=l, case=c, cluster=False)
            
        # Process times for cluster
        for t in times[1]:
            key = (t, l, c, True)
            print(f"Analyzing {key}...")
            data_cache[key] = get_heatmap_data(time=t, local=l, case=c, cluster=True)

print("✅ All data loaded into memory.")

Analyzing (1995, True, 'shape', False)...
Analyzing (14995, True, 'shape', True)...
Analyzing (1995, False, 'shape', False)...
Analyzing (14995, False, 'shape', True)...
✅ All data loaded into memory.


In [10]:
# Change these variables to tweak the look of all plots instantly!
CUSTOM_STEP = 1 
SHOW_SAVE = True

for key, (heatmap_df, title) in data_cache.items():
    t, l, c, is_cluster = key
    
    # Call the fast rendering function
    plot_heatmap(heatmap_df,title,t, local=l, case=c,cluster=is_cluster)
    
    # If working in a notebook, you might want to remove plt.close 
    # to see them all, or keep it to save memory.
    # plt.close()

<Figure size 1000x800 with 0 Axes>

<Figure size 1000x800 with 0 Axes>

<Figure size 1000x800 with 0 Axes>

<Figure size 1000x800 with 0 Axes>